# Numeric-cohort SFT → RLOO：Colab T4 pilot
先到「執行階段 → 變更執行階段類型」選 **T4 GPU**，再由上而下執行。
先完成 numeric cohort 的兩個 epoch SFT，再使用新產生的 adapter 跑 correctness-only RLOO。
2-step smoke 接續到 **總共 50 updates**（400 training completions），前後各 128 個 dev 回覆。
訓練與評估都由模型自然生成；沒有 FSM / repetition penalty / ngram 控制或格式補寫。

先確認下一格 `ADAPTER_SOURCE`。完整說明在 repo 的 `docs/rloo_pilot.md`。
所有輸出保存 Drive，session 到期可續跑；不要讓兩個 sessions 同時寫同一個 RUN_ROOT。
SFT train/dev 強制使用 `numeric_eligible == True`；adapter SHA256 在第一次 prepare 時記錄。
本次使用 `4339742` 的 boxed prompt，重新量測 baseline，不恢復舊格式訓練流程。
先將新 SFT final-model 打包為 `numeric-sft2ep-adapter.zip`，上傳到 Drive 的 `post-train-math-runs` 資料夾。


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

from google.colab import drive
from IPython.display import Markdown, display

drive.mount('/content/drive')
REPO_REF = 'codex/boxed-numeric-rl'
REPO = Path('/content/post-train-math-rloo')
RUN_ROOT = Path('/content/drive/MyDrive/post-train-math-runs/numeric-rloo-sft2ep-v1')
ADAPTER_SOURCE = Path('/content/drive/MyDrive/post-train-math-runs/numeric-sft2ep-adapter.zip')
LOCAL_ADAPTER = Path('/content/rloo-initial-sft')
required = ['adapter_model.safetensors', 'adapter_config.json', 'base_model_source.json',
            'tokenizer.json', 'tokenizer_config.json']
LOCAL_ADAPTER.mkdir(parents=True, exist_ok=True)
if ADAPTER_SOURCE.is_dir():
    for name in required:
        shutil.copy2(ADAPTER_SOURCE / name, LOCAL_ADAPTER / name)
elif ADAPTER_SOURCE.is_file() and ADAPTER_SOURCE.suffix == '.zip':
    import zipfile
    with zipfile.ZipFile(ADAPTER_SOURCE) as archive:
        for name in required:
            (LOCAL_ADAPTER / name).write_bytes(archive.read(name))
else:
    raise FileNotFoundError(f'請把新的 numeric-sft2ep-adapter.zip 放到 {ADAPTER_SOURCE}，或改成新 SFT final-model 目錄。')
with (LOCAL_ADAPTER / 'adapter_model.safetensors').open('rb') as handle:
    digest = hashlib.file_digest(handle, 'sha256').hexdigest()
print('SFT adapter SHA256 (pinned during prepare):', digest)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
revision_file = RUN_ROOT / 'source_revision.json'
if revision_file.exists():
    REPO_REF = json.loads(revision_file.read_text())['git_commit']

def git(*args):
    subprocess.run(['git', *map(str, args)], check=True)

if not REPO.exists():
    git('clone', '--no-checkout', 'https://github.com/ywsu0522/post-train-math.git', REPO)
elif subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Repository 有本地修改；請保留修改並換一個 REPO 目錄。')
git('-C', REPO, 'fetch', 'origin', REPO_REF)
git('-C', REPO, 'checkout', '--detach', 'FETCH_HEAD')
os.chdir(REPO)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if not (REPO / 'src/posttrain_math/rl_pilot.py').is_file():
    raise RuntimeError('此 revision 尚無 RL pilot；請使用提供 notebook 時的新版 branch。')
revision_file.write_text(json.dumps({'git_commit': commit, 'requested_ref': REPO_REF}, indent=2))
print('Pinned source commit:', commit)
print('Verified SFT weights:', digest)

sys.path.insert(0, str(REPO / 'src'))
from posttrain_math.progress import stream_command

SESSION_LOG_DIR = RUN_ROOT / 'session-logs' / datetime.now(UTC).strftime('%Y%m%dT%H%M%S')
def run(command):
    stream_command(list(map(str, command)), log_path=SESSION_LOG_DIR / 'commands.log',
                   status_path=SESSION_LOG_DIR / 'status.json')
print('Session logs:', SESSION_LOG_DIR)

## 1. 安裝與檢查 GPU
使用 repo 的 Python / uv.lock。每個命令串流輸出，安靜時定期 heartbeat；heartbeat 只代表程序未退出。
`doctor` 若不支援目前 driver 會及早報錯；不要在原 run 目錄臨時換 dependency 版本。

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv>=0.12.6'])
run(['uv', 'sync', '--locked', '--group', 'dev'])
run(['uv', 'run', '--locked', '--no-sync', 'python', '-c',
     'import torch; assert torch.cuda.is_available(), "Select a GPU runtime"; print(torch.cuda.get_device_name(0))'])
def cli(*args):
    run(['uv', 'run', '--locked', '--no-sync', 'posttrain-math', *args])
cli('doctor')

## 2. 準備固定 dataset 與命令
下載 pinned OLMo2-1B 與 MATH。RL 使用全部難度/主題的 numeric-eligible train pool；
只做 prompt 長度與去重處理，不按解答長度或模型答對率選题，不把參考解答送入 rollout prompt。
更改 seed、權重或 budget 時須使用新 RUN_ROOT。

In [ ]:
cli('model', 'download')
cli('data', 'download')
cli('data', 'prepare')
OPTIONS = ['--max-steps', '50', '--learning-rate', '1e-6', '--global-batch-size', '8',
           '--max-new-tokens', '512', '--seed', '83']
def pilot(stage):
    command = ['uv', 'run', '--locked', '--no-sync', 'python', '-u', '-m',
               'posttrain_math.rl_pilot', stage, '--output-dir', RUN_ROOT]
    if stage not in {'report', 'package'}:
        command += ['--adapter', LOCAL_ADAPTER, *OPTIONS]
    run(command)
def show_report():
    if (RUN_ROOT / 'report.md').exists():
        display(Markdown((RUN_ROOT / 'report.md').read_text()))
pilot('prepare')
print((RUN_ROOT / 'dataset_manifest.json').read_text())

## 3. SFT baseline（只有評估，不更新權重）
固定抽 64 題 dev greedy，其中固定 16 題每題再做 4 次 sampled。
每筆寫入 Drive，斷線可從下一筆續跑。沒有格式率門檻；完成 baseline 就可進入 RL。

In [ ]:
pilot('baseline')
show_report()

## 4. RLOO smoke：前 2 個 optimizer updates
這是真正的 RL，reward 只有最終數值答對得 1，否則 0。每次更新 2 題 × 4 回覆。
使用原訂 50-step scheduler；step 2 保存 optimizer、scheduler、FP16 scaler 與 RNG。
每批顯示 reward / mixed groups，更新完成後顯示 step / ETA，再顯示 checkpoint saved。

In [ ]:
pilot('smoke')
show_report()

## 5. 繼續到總共 50 updates
從 smoke 的 checkpoint 接續，不重新開始，也不是再加 50 次。
每 10 步保存，保留最近 2 個完整 checkpoint。session 中斷後重跑 setup，再重跑本格即可續訓；
未保存的更新可能重跑，raw logs 會分 attempt 保留，完成的 run 不會重複訓練。

In [ ]:
pilot('train')
show_report()

## 6. RL final 評估與比較
與 SFT baseline 同題、同 seed、同採樣參數與生成長度。
Greedy 與 sampled 分開報告；這是 numeric dev 子集，不是完整 MATH accuracy。先看答對率與 paired gains/losses，
再看 numeric box、token limit 與 sampled mixed groups；不只看訓練 reward。

In [ ]:
pilot('evaluate')
show_report()
print((RUN_ROOT / 'report.json').read_text())

## 狀態與錯誤診斷（可選）
中斷或報錯後可執行；不載入模型、不啟動訓練。若 session 已到期，先 mount Drive。

In [ ]:
for path in sorted((RUN_ROOT / 'steps').glob('*.json')):
    print(path.name, path.read_text()[-5000:])
for path in [RUN_ROOT / 'rl/progress.json', *sorted((RUN_ROOT / 'evals').glob('*/*/progress.json'))]:
    if path.exists():
        print(path.relative_to(RUN_ROOT), path.read_text())
for path in sorted((RUN_ROOT / 'steps').glob('*.log')):
    print('\nLOG:', path.name)
    print('\n'.join(path.read_text(errors='replace').splitlines()[-20:]))

## 7. 下載分析檔給我（成功或失敗都可）
執行下格後，把 **analysis_bundle.zip** 傳回；可另外附有輸出的 notebook。
完整 checkpoint 與 `rl/final-model` 留在 Drive，不必傳權重。

此格只需 Python 與 Drive，可在全新 session **單獨執行**，不需下載模型或安裝訓練套件。
如果之前改了 RUN_ROOT，請把下格預設路徑改成同一個目錄。

In [ ]:
import zipfile
from pathlib import Path

from google.colab import drive, files

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
RUN_ROOT = Path(globals().get('RUN_ROOT', '/content/drive/MyDrive/post-train-math-runs/numeric-rloo-sft2ep-v1'))
if not RUN_ROOT.is_dir():
    raise FileNotFoundError(f'請設定正確的 RUN_ROOT：{RUN_ROOT}')
target = RUN_ROOT / 'analysis_bundle.zip'
temporary = RUN_ROOT / 'analysis_bundle.zip.tmp'
with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_ROOT.rglob('*')):
        relative = path.relative_to(RUN_ROOT)
        if any(p.startswith('checkpoint-') or p == 'final-model' for p in relative.parts):
            continue
        if path.is_file() and (path.suffix in {'.json', '.jsonl', '.md', '.log', '.patch'}
                               or path.name == 'code_snapshot.zip'):
            archive.write(path, relative.as_posix())
temporary.replace(target)
print(f'{target} ({target.stat().st_size:,} bytes)')
files.download(str(target))
print('Keep the full Drive run directory:', RUN_ROOT)